# Silver ProductSubcategory

**Objectif :** charger la table `silver_product_subcategory`.

Cette table est chargée **une seule fois** et ne change pas : aucun historique SCD2 n'est nécessaire. Le notebook :
1. lit le fichier Parquet ;
2. standardise les noms de colonnes en `snake_case` ;
3. ajoute deux colonnes de traçabilité (`load_date`, `sourec_system`) ;
4. vérifie la qualité des données ;
5. écrit la table Delta (`overwrite`, donc relançable sans risque).

## 1. Paramètres

`key` est l'identifiant de la table, **après** conversion en `snake_case` (ex. `ProductSubcategoryID` → `product_subcategory_id`).

In [1]:
import re
from pyspark.sql import functions as F

source_path  = "Files/landing/full/ProductSubcategory"
target_table = "silver_product_subcategory"

key           = "product_subcategory_id"
sourec_system = "adls_landing"

StatementMeta(, ef7c60c2-0000-4042-a9c2-e0122f02e9b3, 3, Finished, Available, Finished, False)

## 2. Fonctions de standardisation des colonnes

`to_snake_case` convertit un nom (`ProductCategoryID` → `product_category_id`). `standardize_columns` l'applique à toutes les colonnes.

In [2]:
def to_snake_case(name):
    name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)   # minuscule -> Majuscule
    name = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", "_", name) # acronyme suivi d'un mot (URLName -> URL_Name)
    return name.lower()


def standardize_columns(df):
    return df.toDF(*[to_snake_case(c) for c in df.columns])

StatementMeta(, ef7c60c2-0000-4042-a9c2-e0122f02e9b3, 4, Finished, Available, Finished, False)

## 3. Fonction d'ajout des colonnes de traçabilité

`load_date` provient du nom du dossier `load_date=YYYY-MM-DD` (déduit automatiquement par Spark). `sourec_system` identifie l'origine des données.

In [3]:
def add_audit_columns(df):
    return (
        df
        .withColumn("load_date", F.to_date("load_date"))
        .withColumn("sourec_system", F.lit(sourec_system))
    )

StatementMeta(, ef7c60c2-0000-4042-a9c2-e0122f02e9b3, 5, Finished, Available, Finished, False)

## 4. Lecture et transformation

Lecture du fichier, puis application des deux fonctions.

In [4]:
raw_data = spark.read.parquet(source_path)
product_subcategory  = add_audit_columns(standardize_columns(raw_data))

print(f"Lignes lues : {product_subcategory.count()}")
display(product_subcategory.limit(10))

StatementMeta(, ef7c60c2-0000-4042-a9c2-e0122f02e9b3, 6, Finished, Available, Finished, False)

Lignes lues : 37


SynapseWidget(Synapse.DataFrame, b50f3f28-a53b-431f-99cf-26551cb18b04)

## 5. Contrôles de qualité

Avant l'écriture, on vérifie que :
- la table n'est pas vide ;
- la colonne clé existe ;
- la clé n'est jamais `NULL` et n'a pas de doublon.

In [5]:
def check_quality(df, key):
    if key not in df.columns:
        raise ValueError(f"Colonne clé '{key}' introuvable. Colonnes disponibles : {df.columns}")

    if df.limit(1).count() == 0:
        raise ValueError("Le fichier source est vide.")

    null_keys  = df.filter(F.col(key).isNull()).count()
    duplicates = df.groupBy(key).count().filter("count > 1").count()

    print(f"Clés NULL      : {null_keys}")
    print(f"Clés en double : {duplicates}")

    if null_keys or duplicates:
        raise ValueError("Échec des contrôles de qualité.")

    print("Contrôles de qualité réussis.")


check_quality(product_subcategory, key)

StatementMeta(, ef7c60c2-0000-4042-a9c2-e0122f02e9b3, 7, Finished, Available, Finished, False)

Clés NULL      : 0
Clés en double : 0
Contrôles de qualité réussis.


## 6. Écriture de la table Silver

Le mode `overwrite` rend le notebook relançable : la table est simplement remplacée par le contenu du fichier source.

In [6]:
(
    product_subcategory
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Table écrite : {target_table}")

StatementMeta(, ef7c60c2-0000-4042-a9c2-e0122f02e9b3, 8, Finished, Available, Finished, False)

Table écrite : silver_product_subcategory


## 7. Consultation du résultat

In [7]:
display(spark.read.table(target_table).orderBy(key))

StatementMeta(, ef7c60c2-0000-4042-a9c2-e0122f02e9b3, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 69f93151-eb6b-444d-96a7-9b138e0c9f48)